# Employee Dashboard - Interactive Jupyter App

Dashboard interaktif untuk filtering dan visualisasi data karyawan.

In [1]:
import os
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set matplotlib untuk interactive mode
%matplotlib inline

In [2]:
# ===== INISIALISASI DATA SAMPLE =====
# Membuat sample data employee
data = {
    'Name': ['Budi Santoso', 'Ani Wijaya', 'Citra Dewi', 'Dedi Kurniawan', 'Eka Putri',
             'Fajar Rahman', 'Gita Sari', 'Hendra Gunawan', 'Indah Lestari', 'Joko Widodo'],
    'Department': ['IT', 'HR', 'IT', 'Finance', 'HR', 'IT', 'Finance', 'IT', 'HR', 'Finance'],
    'Status': ['Permanent', 'Contract', 'Permanent', 'Permanent', 'Contract', 
               'Permanent', 'Contract', 'Permanent', 'Permanent', 'Contract'],
    'Location': ['Jakarta', 'Bandung', 'Jakarta', 'Surabaya', 'Jakarta',
                 'Bandung', 'Jakarta', 'Surabaya', 'Jakarta', 'Bandung'],
    'Gender': ['Male', 'Female', 'Female', 'Male', 'Female',
               'Male', 'Female', 'Male', 'Female', 'Male'],
    'Session': ['Morning', 'Afternoon', 'Morning', 'Morning', 'Afternoon',
                'Morning', 'Afternoon', 'Morning', 'Afternoon', 'Morning'],
    'Salary': [8000000, 6500000, 9500000, 7500000, 6000000,
               10000000, 7000000, 8500000, 6800000, 9000000],
    'Performance_Score': [85, 78, 92, 88, 75, 90, 82, 87, 80, 91],
    'Experience': [5, 3, 7, 4, 2, 8, 3, 6, 4, 9]
}

df = pd.DataFrame(data)
print("Data berhasil dimuat!")
print(f"Total karyawan: {len(df)}")
df.head()

Data berhasil dimuat!
Total karyawan: 10


,Name,Department,Status,Location,Gender,Session,Salary,Performance_Score,Experience
0,Budi Santoso,IT,Permanent,Jakarta,Male,Morning,8000000,85,5
1,Ani Wijaya,HR,Contract,Bandung,Female,Afternoon,6500000,78,3
2,Citra Dewi,IT,Permanent,Jakarta,Female,Morning,9500000,92,7
3,Dedi Kurniawan,Finance,Permanent,Surabaya,Male,Morning,7500000,88,4
4,Eka Putri,HR,Contract,Jakarta,Female,Afternoon,6000000,75,2


In [3]:
# ===== INISIALISASI WIDGETS =====
# Dropdown untuk filter
w_dept = widgets.Dropdown(
    options=['All'] + list(df['Department'].unique()),
    value='All',
    description='Department:'
)

w_status = widgets.Dropdown(
    options=['All'] + list(df['Status'].unique()),
    value='All',
    description='Status:'
)

w_loc = widgets.Dropdown(
    options=['All'] + list(df['Location'].unique()),
    value='All',
    description='Location:'
)

w_gender = widgets.Dropdown(
    options=['All'] + list(df['Gender'].unique()),
    value='All',
    description='Gender:'
)

w_session = widgets.Dropdown(
    options=['All'] + list(df['Session'].unique()),
    value='All',
    description='Session:'
)

# Text input untuk search nama
w_name = widgets.Text(
    value='',
    placeholder='Cari nama...',
    description='Name:'
)

# Slider untuk salary range
w_min_salary = widgets.IntSlider(
    value=int(df['Salary'].min()),
    min=int(df['Salary'].min()),
    max=int(df['Salary'].max()),
    step=100000,
    description='Min Salary:',
    style={'description_width': 'initial'}
)

w_max_salary = widgets.IntSlider(
    value=int(df['Salary'].max()),
    min=int(df['Salary'].min()),
    max=int(df['Salary'].max()),
    step=100000,
    description='Max Salary:',
    style={'description_width': 'initial'}
)

# Button untuk export
btn_export = widgets.Button(
    description='Export to CSV',
    button_style='success',
    icon='download'
)

# Output widget untuk menampilkan hasil
out = widgets.Output()

print("Widgets berhasil dibuat!")

Widgets berhasil dibuat!


In [4]:
# ===== FUNGSI-FUNGSI =====
def apply_filters():
    dff = df.copy()
    
    if w_dept.value != "All":
        dff = dff[dff["Department"] == w_dept.value]
        
    if w_status.value != "All":
        dff = dff[dff["Status"] == w_status.value]
        
    if w_loc.value != "All":
        dff = dff[dff["Location"] == w_loc.value]
        
    if w_gender.value != "All":
        dff = dff[dff["Gender"] == w_gender.value]
        
    if w_session.value != "All":
        dff = dff[dff["Session"] == w_session.value]
        
    # salary range
    dff = dff[(dff["Salary"] >= w_min_salary.value) & (dff["Salary"] <= w_max_salary.value)]
    
    # search nama
    if w_name.value.strip():
        dff = dff[dff["Name"].str.contains(w_name.value.strip(), case=False, na=False)]
        
    return dff

def render_dashboard(_=None):
    with out:
        clear_output(wait=True)
        dff = apply_filters()
        
        # METRICS
        total = len(dff)
        avg_salary = dff["Salary"].mean() if total > 0 else 0
        med_salary = dff["Salary"].median() if total > 0 else 0
        avg_perf = dff["Performance_Score"].mean() if total > 0 else 0
        avg_exp = dff["Experience"].mean() if total > 0 else 0
        
        print("=" * 70)
        print("=== EMPLOYEE DASHBOARD (JUPYTER APP) ===")
        print("=" * 70)
        print(f"Total Karyawan (hasil filter): {total}")
        print(f"Avg Salary: Rp {avg_salary:,.0f}" if total else "Avg Salary: -")
        print(f"Median Salary: Rp {med_salary:,.0f}" if total else "Median Salary: -")
        print(f"Avg Performance Score: {avg_perf:.2f}" if total else "Avg Performance Score: -")
        print(f"Avg Experience: {avg_exp:.2f} tahun" if total else "Avg Experience: -")
        print("=" * 70)
        print()
        
        # TABLE
        if total > 0:
            display(dff)
            
            # CHARTS
            # Chart 1: count per department
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            dff["Department"].value_counts().plot(kind="bar", ax=axes[0], color='steelblue')
            axes[0].set_title("Jumlah Karyawan per Department", fontsize=14, fontweight='bold')
            axes[0].set_xlabel("Department")
            axes[0].set_ylabel("Count")
            axes[0].tick_params(axis='x', rotation=45)
            
            # Chart 2: salary distribution (sorted line chart)
            dff["Salary"].dropna().sort_values().reset_index(drop=True).plot(ax=axes[1], color='green', linewidth=2)
            axes[1].set_title("Distribusi Salary (sorted)", fontsize=14, fontweight='bold')
            axes[1].set_xlabel("Index (urut)")
            axes[1].set_ylabel("Salary (Rp)")
            axes[1].grid(True, alpha=0.3)
            
            # Chart 3: status pie
            dff["Status"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[2], colors=['#ff9999','#66b3ff'])
            axes[2].set_title("Komposisi Status", fontsize=14, fontweight='bold')
            axes[2].set_ylabel("")
            
            plt.tight_layout()
            plt.show()
        else:
            print("⚠️ Tidak ada data yang sesuai dengan filter.")

def export_csv(_=None):
    dff = apply_filters()
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_name = f"filtered_employee_{ts}.csv"
    dff.to_csv(out_name, index=False)
    
    with out:
        print(f"\n✅ File export berhasil: {out_name}")
        print(f"📁 Lokasi: {os.path.abspath(out_name)}")

print("Fungsi-fungsi berhasil didefinisikan!")

Fungsi-fungsi berhasil didefinisikan!


In [5]:
# ===== SETUP EVENT HANDLERS =====
btn_export.on_click(export_csv)

# auto update ketika widget berubah
for w in [w_dept, w_status, w_loc, w_gender, w_session, w_name, w_min_salary, w_max_salary]:
    w.observe(render_dashboard, names="value")

print("Event handlers berhasil di-setup!")

Event handlers berhasil di-setup!


In [6]:
# ===== TAMPILAN DASHBOARD =====
print("🚀 Launching Employee Dashboard...\n")

ui_row1 = widgets.HBox([w_dept, w_status, w_loc])
ui_row2 = widgets.HBox([w_gender, w_session])
ui_row3 = widgets.HBox([w_name])
ui_row4 = widgets.HBox([w_min_salary, w_max_salary])
ui_row5 = widgets.HBox([btn_export])

display(ui_row1, ui_row2, ui_row3, ui_row4, ui_row5, out)
render_dashboard()

🚀 Launching Employee Dashboard...



Output()